# Sistema de Recomendação de Transporte Público — API Olho Vivo

**Trabalho em grupo | Python | Jupyter Notebook**

## Objetivo
Recomendar uma viagem direta de ônibus entre duas coordenadas geográficas, utilizando a base GTFS para identificar viagens que atendam à origem e ao destino e a API Olho Vivo da SPTrans para consultar dados operacionais e a previsão oficial de chegada, quando disponível.

## Estratégia adotada
1. Validar as coordenadas e o raio de busca (1 a 1.000 metros).
2. Carregar a base GTFS fornecida pelo grupo.
3. Encontrar paradas próximas à origem e ao destino usando a distância de Haversine.
4. Cruzar os registros de `stop_times.txt` pelo mesmo `trip_id`, exigindo que a sequência do embarque seja anterior à do desembarque. Isso garante uma viagem direta, sem baldeação.
5. Classificar as alternativas pela soma das distâncias a pé até o embarque e depois do desembarque.
6. Consultar a API Olho Vivo para confirmar a linha/sentido e obter a previsão no ponto de embarque, quando houver correspondência geográfica suficiente.
7. Exibir o resultado e um mapa interativo com origem, destino e paradas do trecho.

**Importante:** GTFS é a base estática de linhas, viagens e sequência de paradas. A API Olho Vivo fornece dados operacionais em tempo real. A previsão oficial só será exibida quando a API retornar esse dado; distância GPS não será apresentada como se fosse ETA.

## 1. Instalação e arquivos necessários

Na pasta do projeto, mantenha:
- `Sistema_Transporte_Olho_Vivo_ENTREGA.ipynb`
- `_gtfs.zip`
- `.env` (somente no computador local; não enviar ao GitHub)
- `.gitignore`

Instale as dependências em um terminal. O arquivo `.env` deve ficar nesta mesma pasta e conter `SPTRANS_TOKEN=SEU_TOKEN` (não compartilhe nem envie ao GitHub).\n\nInstale as dependências em um terminal:

```bash
pip install pandas requests python-dotenv folium ipython
```

O token deve estar no arquivo `.env`, no formato:

```text
SPTRANS_TOKEN=SEU_TOKEN_AQUI
```

O arquivo `.env` não deve ser publicado. O repositório deve conter apenas `.env.example`.

In [ ]:
from pathlib import Path
import os
import math
import zipfile
import requests
import pandas as pd
import folium
from dotenv import load_dotenv
from IPython.display import display
from IPython.display import Markdown

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

PASTA_PROJETO = Path.cwd()
ARQUIVO_GTFS = PASTA_PROJETO / "_gtfs.zip"
PASTA_GTFS = PASTA_PROJETO / "dados_gtfs"
BASE_URL = "https://api.olhovivo.sptrans.com.br/v2.1"

print("Ambiente carregado.")
print("Pasta de trabalho:", PASTA_PROJETO)
print("Arquivo GTFS encontrado:", ARQUIVO_GTFS.exists())

## 2. Configuração da consulta

As coordenadas abaixo são exemplos de teste na cidade de São Paulo. Substitua-as pelas coordenadas escolhidas pelo grupo para a demonstração. O raio deve ficar entre 1 e 1.000 metros.

In [ ]:
# Exemplo de coordenadas em São Paulo (substitua se desejarem)
origem = {
    "latitude": -23.5465,
    "longitude": -46.6525
}

destino = {
    "latitude": -23.5505,
    "longitude": -46.6333
}

RAIO_METROS = 500
LIMITE_RECOMENDACOES = 5

def validar_coordenada(ponto, nome):
    if not isinstance(ponto, dict):
        raise ValueError(f"{nome}: informe latitude e longitude em um dicionário.")
    if "latitude" not in ponto or "longitude" not in ponto:
        raise ValueError(f"{nome}: latitude ou longitude ausente.")
    lat = float(ponto["latitude"])
    lon = float(ponto["longitude"])
    if not (-90 <= lat <= 90):
        raise ValueError(f"{nome}: latitude fora do intervalo [-90, 90].")
    if not (-180 <= lon <= 180):
        raise ValueError(f"{nome}: longitude fora do intervalo [-180, 180].")
    return {"latitude": lat, "longitude": lon}

def validar_entrada(origem, destino, raio):
    origem_ok = validar_coordenada(origem, "Origem")
    destino_ok = validar_coordenada(destino, "Destino")
    raio = float(raio)
    if not (1 <= raio <= 1000):
        raise ValueError("O raio deve estar entre 1 e 1.000 metros.")
    return origem_ok, destino_ok, raio

origem, destino, RAIO_METROS = validar_entrada(origem, destino, RAIO_METROS)
print("Origem validada:", origem)
print("Destino validado:", destino)
print("Raio de busca:", RAIO_METROS, "metros")

## 3. Carregamento e validação dos dados GTFS

O arquivo compactado é extraído localmente. São utilizados `stops.txt`, `routes.txt`, `trips.txt` e `stop_times.txt`. `shapes.txt` é utilizado para desenhar o percurso no mapa quando a viagem possuir geometria cadastrada.

In [ ]:
def carregar_gtfs(arquivo_zip=ARQUIVO_GTFS, pasta_destino=PASTA_GTFS):
    if not Path(arquivo_zip).exists():
        raise FileNotFoundError(
            "Arquivo _gtfs.zip não encontrado. Coloque-o na mesma pasta do notebook."
        )
    pasta_destino.mkdir(parents=True, exist_ok=True)
    arquivos_necessarios = [
        "stops.txt", "routes.txt", "trips.txt", "stop_times.txt"
    ]
    with zipfile.ZipFile(arquivo_zip, "r") as zf:
        nomes = set(Path(n).name for n in zf.namelist())
        ausentes = [n for n in arquivos_necessarios if n not in nomes]
        if ausentes:
            raise ValueError(f"GTFS incompleto. Arquivos ausentes: {ausentes}")
        zf.extractall(pasta_destino)

    def ler(nome, **kwargs):
        caminho = pasta_destino / nome
        if not caminho.exists():
            # Alguns ZIPs colocam os arquivos dentro de uma subpasta.
            encontrados = list(pasta_destino.rglob(nome))
            if not encontrados:
                raise FileNotFoundError(f"Não foi possível localizar {nome}.")
            caminho = encontrados[0]
        return pd.read_csv(caminho, dtype=str, encoding="utf-8-sig", **kwargs)

    dados = {
        "paradas": ler("stops.txt"),
        "linhas": ler("routes.txt"),
        "viagens": ler("trips.txt"),
        "horarios": ler("stop_times.txt"),
    }
    shapes = list(pasta_destino.rglob("shapes.txt"))
    dados["formas"] = ler("shapes.txt") if shapes else pd.DataFrame()
    return dados

try:
    gtfs = carregar_gtfs()
    gtfs_paradas = gtfs["paradas"]
    gtfs_linhas = gtfs["linhas"]
    gtfs_viagens = gtfs["viagens"]
    gtfs_horarios = gtfs["horarios"]
    gtfs_shapes = gtfs["formas"]

    print("GTFS carregado com sucesso.")
    print("Paradas:", len(gtfs_paradas))
    print("Linhas:", len(gtfs_linhas))
    print("Viagens:", len(gtfs_viagens))
    print("Registros em stop_times:", len(gtfs_horarios))
    print("Registros em shapes:", len(gtfs_shapes))
except Exception as erro:
    print("Falha ao carregar GTFS:", erro)
    raise

## 4. Cálculo geográfico e busca de paradas próximas

A fórmula de Haversine calcula a distância aproximada em metros entre duas coordenadas na superfície terrestre. O algoritmo utiliza o raio definido pelo grupo para selecionar as paradas candidatas.

In [ ]:
def calcular_distancia(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(
        math.radians, [float(lat1), float(lon1), float(lat2), float(lon2)]
    )
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    return 2 * 6371000 * math.asin(math.sqrt(a))

def preparar_paradas(paradas):
    obrigatorias = {"stop_id", "stop_name", "stop_lat", "stop_lon"}
    faltantes = obrigatorias - set(paradas.columns)
    if faltantes:
        raise ValueError(f"Colunas ausentes em stops.txt: {sorted(faltantes)}")
    df = paradas.copy()
    df["stop_lat"] = pd.to_numeric(df["stop_lat"], errors="coerce")
    df["stop_lon"] = pd.to_numeric(df["stop_lon"], errors="coerce")
    return df.dropna(subset=["stop_lat", "stop_lon"])

gtfs_paradas = preparar_paradas(gtfs_paradas)

def buscar_paradas_proximas(ponto, paradas, raio):
    df = paradas.copy()
    df["distancia_m"] = df.apply(
        lambda r: calcular_distancia(
            ponto["latitude"], ponto["longitude"],
            r["stop_lat"], r["stop_lon"]
        ),
        axis=1
    )
    return (
        df[df["distancia_m"] <= raio]
        .sort_values("distancia_m")
        .reset_index(drop=True)
    )

paradas_origem = buscar_paradas_proximas(origem, gtfs_paradas, RAIO_METROS)
paradas_destino = buscar_paradas_proximas(destino, gtfs_paradas, RAIO_METROS)

print("Paradas próximas da origem:", len(paradas_origem))
print("Paradas próximas do destino:", len(paradas_destino))

display(paradas_origem[["stop_id", "stop_name", "distancia_m"]].head(10))
display(paradas_destino[["stop_id", "stop_name", "distancia_m"]].head(10))

## 5. Identificação de viagens diretas

A comparação é feita pelo identificador `trip_id`. O mesmo ônibus/viagem precisa atender a uma parada próxima da origem e outra próxima do destino. A sequência (`stop_sequence`) do embarque deve ser menor que a do desembarque. Assim, o algoritmo não sugere baldeações.

In [ ]:
def buscar_viagens_diretas(paradas_origem, paradas_destino, horarios, viagens, linhas):
    if paradas_origem.empty or paradas_destino.empty:
        return pd.DataFrame()

    obrigatorias = {"trip_id", "stop_id", "stop_sequence", "arrival_time", "departure_time"}
    faltantes = obrigatorias - set(horarios.columns)
    if faltantes:
        raise ValueError(f"Colunas ausentes em stop_times.txt: {sorted(faltantes)}")

    h = horarios.copy()
    h["stop_sequence_num"] = pd.to_numeric(h["stop_sequence"], errors="coerce")
    h = h.dropna(subset=["stop_sequence_num"])

    emb = h.merge(
        paradas_origem[["stop_id", "stop_name", "distancia_m"]],
        on="stop_id", how="inner"
    ).rename(columns={
        "stop_id": "stop_id_emb",
        "stop_name": "parada_embarque",
        "distancia_m": "distancia_origem_m",
        "stop_sequence_num": "sequencia_embarque",
        "arrival_time": "chegada_embarque_tabela",
        "departure_time": "saida_embarque_tabela"
    })

    des = h.merge(
        paradas_destino[["stop_id", "stop_name", "distancia_m"]],
        on="stop_id", how="inner"
    ).rename(columns={
        "stop_id": "stop_id_des",
        "stop_name": "parada_desembarque",
        "distancia_m": "distancia_destino_m",
        "stop_sequence_num": "sequencia_desembarque",
        "arrival_time": "chegada_desembarque_tabela"
    })

    combinacoes = emb.merge(des, on="trip_id", suffixes=("_emb", "_des"))
    combinacoes = combinacoes[
        combinacoes["sequencia_embarque"] < combinacoes["sequencia_desembarque"]
    ].copy()

    if combinacoes.empty:
        return pd.DataFrame()

    colunas_viagens = [
        c for c in ["trip_id", "route_id", "service_id", "trip_headsign",
                    "direction_id", "shape_id"] if c in viagens.columns
    ]
    combinacoes = combinacoes.merge(
        viagens[colunas_viagens].drop_duplicates("trip_id"),
        on="trip_id", how="left"
    )
    colunas_linhas = [
        c for c in ["route_id", "route_short_name", "route_long_name"]
        if c in linhas.columns
    ]
    combinacoes = combinacoes.merge(
        linhas[colunas_linhas].drop_duplicates("route_id"),
        on="route_id", how="left"
    )
    combinacoes["distancia_total_caminhada_m"] = (
        combinacoes["distancia_origem_m"] + combinacoes["distancia_destino_m"]
    )

    combinacoes = combinacoes.sort_values(
        ["distancia_total_caminhada_m", "distancia_origem_m", "distancia_destino_m"]
    )
    combinacoes = combinacoes.drop_duplicates(
        ["trip_id", "stop_id_emb", "stop_id_des"]
    )
    return combinacoes.reset_index(drop=True)

recomendacoes = buscar_viagens_diretas(
    paradas_origem, paradas_destino,
    gtfs_horarios, gtfs_viagens, gtfs_linhas
)

print("Viagens diretas encontradas:", len(recomendacoes))
if recomendacoes.empty:
    print("Nenhuma viagem direta encontrada dentro do raio configurado.")
else:
    colunas_exibir = [
        c for c in [
            "route_short_name", "trip_headsign", "direction_id",
            "parada_embarque", "parada_desembarque",
            "distancia_origem_m", "distancia_destino_m",
            "distancia_total_caminhada_m",
            "saida_embarque_tabela", "chegada_desembarque_tabela"
        ] if c in recomendacoes.columns
    ]
    display(recomendacoes[colunas_exibir].head(LIMITE_RECOMENDACOES).round(1))

## 6. Autenticação e integração com a API Olho Vivo

O token é lido do `.env` e não é impresso. A autenticação é feita por `POST /Login/Autenticar`. Em seguida, o sistema consulta `GET /Linha/Buscar`, `GET /Parada/BuscarParadasPorLinha` e `GET /Previsao`.

A API da SPTrans identifica linhas por código próprio e por sentido. Para evitar associação incorreta, o código tenta relacionar os pontos GTFS de embarque/desembarque aos pontos retornados pela API por proximidade geográfica. Se não houver correspondência suficiente, a recomendação GTFS permanece disponível, mas a previsão oficial é informada como indisponível.

In [ ]:
ARQUIVO_ENV = PASTA_PROJETO / ".env"
load_dotenv(dotenv_path=ARQUIVO_ENV, override=True)
TOKEN_SPTRANS = os.getenv("SPTRANS_TOKEN")

sessao_sptrans = requests.Session()
sessao_sptrans.headers.update({"Accept": "application/json"})

def autenticar_sptrans():
    if not TOKEN_SPTRANS:
        raise RuntimeError(
            "Token não encontrado. Crie o arquivo .env com SPTRANS_TOKEN=SEU_TOKEN."
        )
    url = f"{BASE_URL}/Login/Autenticar"
    resposta = sessao_sptrans.post(
        url, params={"token": TOKEN_SPTRANS}, timeout=30
    )
    resposta.raise_for_status()
    autenticado = resposta.text.strip().lower() == "true"
    if not autenticado:
        raise RuntimeError("A SPTrans não confirmou a autenticação. Verifique o token.")
    return True

def get_sptrans(endpoint, params=None):
    resposta = sessao_sptrans.get(
        f"{BASE_URL}/{endpoint.lstrip('/')}",
        params=params or {},
        timeout=30
    )
    resposta.raise_for_status()
    try:
        return resposta.json()
    except ValueError as exc:
        raise RuntimeError("A API retornou conteúdo que não é JSON válido.") from exc

try:
    autenticar_sptrans()
    print("Autenticação SPTrans realizada com sucesso.")
except Exception as erro:
    print("Não foi possível autenticar na SPTrans:", erro)

## 7. Validação das linhas candidatas e previsão oficial

O sistema consulta as linhas candidatas pelo letreiro na API Olho Vivo. A documentação da SPTrans informa que a consulta de paradas contempla atualmente apenas paradas de corredores; por isso, ela não deve ser usada como filtro obrigatório para linhas comuns. A ordem do trajeto é validada pelo GTFS (`trip_id` e `stop_sequence`).

Para tentar obter a previsão oficial, o sistema consulta `/Previsao` usando o código da parada de embarque do GTFS e o código de linha retornado por `/Linha/Buscar`. A API pode não reconhecer esse código de parada; nesse caso, a recomendação GTFS é mantida, mas o sistema informa claramente que não há ETA oficial. O sentido (`sl`) é mostrado quando retornado pela SPTrans.



In [ ]:
def campo(objeto, *nomes, padrao=None):
    for nome in nomes:
        if isinstance(objeto, dict) and nome in objeto and objeto[nome] is not None:
            return objeto[nome]
    return padrao


def normalizar_letreiro(valor):
    return str(valor or "").strip().upper().replace(" ", "")


def buscar_linhas_api(letreiro_gtfs):
    """Busca linhas no Olho Vivo e filtra pelo letreiro completo do GTFS."""
    letreiro = normalizar_letreiro(letreiro_gtfs)
    parte_numero = letreiro.split("-")[0]
    respostas = []
    for termo in dict.fromkeys([parte_numero, letreiro]):
        dados = get_sptrans("Linha/Buscar", {"termosBusca": termo})
        if isinstance(dados, list):
            respostas.extend(dados)

    linhas = {}
    for item in respostas:
        codigo = campo(item, "cl", "CodigoLinha")
        lt = campo(item, "lt", "Letreiro")
        tl = campo(item, "tl", "Tipo")
        # v2.1 separa o letreiro (lt) do tipo/atendimento (tl).
        letreiro_api = normalizar_letreiro(f"{lt}-{tl}") if tl is not None else normalizar_letreiro(lt)
        if codigo is not None and (letreiro_api == letreiro or normalizar_letreiro(lt) == parte_numero):
            linhas[str(codigo)] = item
    return list(linhas.values())


def consultar_previsao_no_ponto_gtfs(codigo_parada, codigo_linha):
    """Consulta a previsão usando o código da parada GTFS como código SPTrans.

    Os IDs de parada da base GTFS fornecida são testados diretamente no endpoint
    oficial. Se a SPTrans não reconhecer o código, o sistema informa indisponibilidade.
    """
    return get_sptrans(
        "Previsao",
        {"codigoParada": str(codigo_parada), "codigoLinha": str(codigo_linha)}
    )


def consultar_api_para_viagem(viagem):
    resultado = {
        "linha_api": None,
        "previsoes": [],
        "mensagem": "",
        "horario_referencia_api": None
    }
    letreiro = str(viagem.get("route_short_name", "")).strip()
    try:
        linhas = buscar_linhas_api(letreiro)
        if not linhas:
            resultado["mensagem"] = (
                f"A SPTrans não retornou linha correspondente ao letreiro {letreiro}."
            )
            return resultado

        codigo_parada_gtfs = str(viagem["stop_id_emb"])
        erros = []
        for linha in linhas:
            codigo_linha = campo(linha, "cl", "CodigoLinha")
            if codigo_linha is None:
                continue
            try:
                dados = consultar_previsao_no_ponto_gtfs(codigo_parada_gtfs, codigo_linha)
            except requests.RequestException as erro:
                erros.append(str(erro))
                continue
            except RuntimeError as erro:
                erros.append(str(erro))
                continue

            ponto = campo(dados, "p", padrao={})
            linhas_prev = campo(ponto, "l", padrao=[])
            for linha_prev in linhas_prev:
                codigo_prev = campo(linha_prev, "cl")
                if str(codigo_prev) != str(codigo_linha):
                    continue
                resultado["linha_api"] = linha_prev
                resultado["horario_referencia_api"] = campo(dados, "hr")
                for veiculo in campo(linha_prev, "vs", padrao=[]):
                    resultado["previsoes"].append({
                        "linha": campo(linha_prev, "c", padrao=letreiro),
                        "sentido_sptrans": campo(linha_prev, "sl"),
                        "prefixo_onibus": campo(veiculo, "p"),
                        "previsao_chegada": campo(veiculo, "t"),
                        "acessivel": campo(veiculo, "a"),
                        "atualizacao": campo(veiculo, "ta")
                    })
                if not resultado["previsoes"]:
                    resultado["mensagem"] = (
                        "A linha foi localizada na API, mas não há ônibus com previsão "
                        "disponível neste ponto neste momento."
                    )
                return resultado

        if erros:
            resultado["mensagem"] = (
                "A SPTrans encontrou a linha, mas não foi possível consultar a previsão "
                "no código da parada GTFS. Detalhe: " + " | ".join(erros[:2])
            )
        else:
            resultado["mensagem"] = (
                "A SPTrans encontrou a linha, mas não reconheceu o código da parada "
                "GTFS como ponto válido para previsão. A viagem GTFS permanece como "
                "recomendação, sem ETA oficial."
            )
        return resultado

    except requests.RequestException as erro:
        resultado["mensagem"] = f"Falha de comunicação com a API: {erro}"
        return resultado
    except (KeyError, IndexError, TypeError, ValueError) as erro:
        resultado["mensagem"] = f"Resposta incompleta ou inconsistente da API: {erro}"
        return resultado


def consultar_api_candidatas(recomendacoes, max_linhas=15):
    """Valida linhas candidatas na API; usa GTFS para a ordem do trajeto."""
    if recomendacoes.empty:
        return None, None

    vistos = set()
    mensagens = []
    quantidade = 0
    for _, viagem in recomendacoes.iterrows():
        letreiro = str(viagem.get("route_short_name", "")).strip()
        if not letreiro or letreiro.lower() == "nan" or letreiro in vistos:
            continue
        vistos.add(letreiro)
        if quantidade >= max_linhas:
            break
        quantidade += 1
        consulta = consultar_api_para_viagem(viagem)
        if consulta.get("linha_api"):
            return viagem, consulta
        mensagens.append(f"{letreiro}: {consulta.get('mensagem', 'sem retorno')}")

    return recomendacoes.iloc[0], {
        "linha_api": None,
        "previsoes": [],
        "mensagem": (
            f"Foram consultadas {quantidade} linhas candidatas. "
            + " | ".join(mensagens[:4])
        )
    }


if recomendacoes.empty:
    melhor_viagem_api = None
    consulta_api = None
    print("A consulta à API será realizada após existir uma recomendação.")
else:
    melhor_viagem_api, consulta_api = consultar_api_candidatas(recomendacoes)
    if consulta_api and consulta_api.get("linha_api"):
        print("Linha/sentido retornados pela SPTrans:", consulta_api["linha_api"])
    if consulta_api and consulta_api.get("previsoes"):
        display(pd.DataFrame(consulta_api["previsoes"]))
    elif consulta_api:
        print("Previsão oficial:", consulta_api["mensagem"])


## 8. Mapa interativo da viagem recomendada

O mapa apresenta origem, ponto de embarque, pontos intermediários da viagem, ponto de desembarque e destino. A linha do trajeto é desenhada com `shapes.txt`, quando a geometria está disponível.

In [ ]:
def gerar_mapa_viagem(viagem, origem, destino):
    centro = [
        (origem["latitude"] + destino["latitude"]) / 2,
        (origem["longitude"] + destino["longitude"]) / 2
    ]
    mapa = folium.Map(location=centro, zoom_start=14, control_scale=True)

    folium.Marker(
        [origem["latitude"], origem["longitude"]],
        tooltip="Origem",
        popup="Origem informada",
        icon=folium.Icon(color="green", icon="play")
    ).add_to(mapa)

    parada_emb = gtfs_paradas[
        gtfs_paradas["stop_id"].astype(str) == str(viagem["stop_id_emb"])
    ].iloc[0]
    parada_des = gtfs_paradas[
        gtfs_paradas["stop_id"].astype(str) == str(viagem["stop_id_des"])
    ].iloc[0]

    folium.Marker(
        [parada_emb["stop_lat"], parada_emb["stop_lon"]],
        tooltip="Embarque",
        popup=f"Embarque: {parada_emb['stop_name']}",
        icon=folium.Icon(color="blue", icon="bus")
    ).add_to(mapa)

    trecho_paradas = gtfs_horarios[
        (gtfs_horarios["trip_id"].astype(str) == str(viagem["trip_id"]))
    ].copy()
    trecho_paradas["seq_num"] = pd.to_numeric(
        trecho_paradas["stop_sequence"], errors="coerce"
    )
    trecho_paradas = trecho_paradas[
        (trecho_paradas["seq_num"] >= float(viagem["sequencia_embarque"]))
        & (trecho_paradas["seq_num"] <= float(viagem["sequencia_desembarque"]))
    ].sort_values("seq_num")

    trecho_paradas = trecho_paradas.merge(
        gtfs_paradas[["stop_id", "stop_name", "stop_lat", "stop_lon"]],
        on="stop_id", how="left"
    )

    for _, p in trecho_paradas.iterrows():
        if str(p["stop_id"]) in [str(viagem["stop_id_emb"]), str(viagem["stop_id_des"])]:
            continue
        if pd.notna(p["stop_lat"]) and pd.notna(p["stop_lon"]):
            folium.CircleMarker(
                [p["stop_lat"], p["stop_lon"]],
                radius=4, color="darkblue", fill=True,
                popup=f"Parada intermediária: {p['stop_name']}"
            ).add_to(mapa)

    folium.Marker(
        [parada_des["stop_lat"], parada_des["stop_lon"]],
        tooltip="Desembarque",
        popup=f"Desembarque: {parada_des['stop_name']}",
        icon=folium.Icon(color="red", icon="flag")
    ).add_to(mapa)

    folium.Marker(
        [destino["latitude"], destino["longitude"]],
        tooltip="Destino",
        popup="Destino informado",
        icon=folium.Icon(color="darkred", icon="stop")
    ).add_to(mapa)

    shape_id = viagem.get("shape_id")
    if not gtfs_shapes.empty and pd.notna(shape_id) and "shape_id" in gtfs_shapes.columns:
        geometria = gtfs_shapes[
            gtfs_shapes["shape_id"].astype(str) == str(shape_id)
        ].copy()
        if not geometria.empty:
            geometria["seq"] = pd.to_numeric(geometria["shape_pt_sequence"], errors="coerce")
            geometria["lat"] = pd.to_numeric(geometria["shape_pt_lat"], errors="coerce")
            geometria["lon"] = pd.to_numeric(geometria["shape_pt_lon"], errors="coerce")
            geometria = geometria.dropna(subset=["seq", "lat", "lon"]).sort_values("seq")
            coords = geometria[["lat", "lon"]].values.tolist()
            if len(coords) >= 2:
                folium.PolyLine(
                    coords, color="blue", weight=4, opacity=0.65,
                    tooltip=f"Trajeto da linha {viagem.get('route_short_name', '')}"
                ).add_to(mapa)

    limites = [
        [origem["latitude"], origem["longitude"]],
        [destino["latitude"], destino["longitude"]],
        [parada_emb["stop_lat"], parada_emb["stop_lon"]],
        [parada_des["stop_lat"], parada_des["stop_lon"]]
    ]
    mapa.fit_bounds(limites)
    return mapa

if recomendacoes.empty:
    mapa_viagem = None
    print("Mapa não gerado: nenhuma viagem direta foi encontrada.")
else:
    viagem_mapa = melhor_viagem_api if "melhor_viagem_api" in globals() and melhor_viagem_api is not None else recomendacoes.iloc[0]
    mapa_viagem = gerar_mapa_viagem(viagem_mapa, origem, destino)
    display(mapa_viagem)

## 9. Resultado consolidado

A classificação principal considera a menor distância total de caminhada entre origem/embarque e desembarque/destino. O horário do GTFS é horário de tabela, não previsão em tempo real. A previsão oficial é exibida separadamente, apenas quando retornada pela API.

In [ ]:
def apresentar_resultado(recomendacoes, consulta_api=None):
    if recomendacoes.empty:
        print("Não há recomendação para os parâmetros informados.")
        return

    v = melhor_viagem_api if "melhor_viagem_api" in globals() and melhor_viagem_api is not None else recomendacoes.iloc[0]
    print("=" * 68)
    print("RECOMENDAÇÃO DE TRANSPORTE PÚBLICO")
    print("=" * 68)
    print("Linha GTFS:", v.get("route_short_name"))
    print("Destino/letreiro da viagem:", v.get("trip_headsign"))
    print("Sentido GTFS:", v.get("direction_id"))
    if consulta_api and consulta_api.get("linha_api"):
        print("Sentido informado pela SPTrans:", consulta_api["linha_api"].get("sl", consulta_api["linha_api"].get("Sentido")))
        print("Terminal de origem (SPTrans):", consulta_api["linha_api"].get("tp"))
        print("Terminal de destino (SPTrans):", consulta_api["linha_api"].get("ts"))
    print("Embarque:", v.get("parada_embarque"))
    print("Desembarque:", v.get("parada_desembarque"))
    print(f"Distância a pé até o embarque: {v.get('distancia_origem_m'):.1f} m")
    print(f"Distância a pé após desembarque: {v.get('distancia_destino_m'):.1f} m")
    print(f"Caminhada total: {v.get('distancia_total_caminhada_m'):.1f} m")
    print("Horário de tabela no embarque:", v.get("saida_embarque_tabela"))
    print("Horário de tabela no desembarque:", v.get("chegada_desembarque_tabela"))

    if consulta_api:
        if consulta_api.get("previsoes"):
            print("\nPREVISÃO OFICIAL SPTRANS")
            for prev in consulta_api["previsoes"]:
                print(
                    f"Ônibus {prev.get('prefixo_onibus')}: "
                    f"previsão {prev.get('previsao_chegada')}"
                )
            print("Referência da API:", consulta_api.get("horario_referencia_api"))
        else:
            print("\nPREVISÃO OFICIAL SPTRANS: indisponível nesta consulta.")
            print(consulta_api.get("mensagem", "Sem informação adicional."))
    else:
        print("\nPREVISÃO OFICIAL SPTRANS: não consultada.")

apresentar_resultado(recomendacoes, consulta_api)

## 10. Tratamento de erros e validações

O projeto contempla:
- Coordenadas fora dos limites geográficos ou campos ausentes.
- Raio fora do intervalo permitido.
- Arquivo GTFS ausente ou incompleto.
- Colunas necessárias ausentes nos arquivos GTFS.
- Ausência de paradas próximas ou de viagem direta.
- Token ausente ou falha de autenticação.
- Erros HTTP, respostas não JSON ou dados inconsistentes da API.
- Ausência de previsão oficial, sem fabricar um tempo estimado.

## 11. Fluxograma do algoritmo

```text
INÍCIO
  |
  v
Receber origem, destino e raio
  |
  v
Validar coordenadas e raio
  |
  v
Carregar arquivos GTFS
  |
  v
Calcular distâncias (Haversine)
  |
  v
Encontrar paradas próximas da origem e destino
  |
  v
Cruzar stop_times pelo trip_id
  |
  v
Embarque antes do desembarque?
  |                 |
 não               sim
  |                 |
Descartar      Classificar viagens
                    |
                    v
             Selecionar melhor opção
                    |
                    v
          Autenticar na API Olho Vivo
                    |
                    v
       Confirmar linha e paradas na API
                    |
                    v
       Consultar previsão no embarque
                    |
                    v
       Exibir resultado e mapa Folium
                    |
                    v
                   FIM
```

## 12. Conclusão

A solução integra dados estáticos GTFS com dados operacionais da API Olho Vivo. O algoritmo identifica uma viagem direta, calcula as distâncias de acesso, seleciona a alternativa com menor caminhada e apresenta o trajeto em mapa. A previsão oficial depende da disponibilidade e da correspondência dos dados retornados pela API, sendo tratada explicitamente quando não estiver disponível.